# The Determinants of Turnout

In [1]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [3]:
%%stata

import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date
gen day = date(date, "YMD")
format day %td
gen month = month(day)
gen quarter = quarter(day)

gen year = year(day)
encode gecko_id, gen(token)
encode space,    gen(dao)

* Dependent variables
local dep non_whale_turnout whale_turnout

replace concensus = concensus * have_discussion
replace professionalism = professionalism * have_discussion

* Independent variables
local complexity multi_choices weighted ranked_choice quorum delegation
local discussion_char concensus professionalism
local topic protocol_security treasury_expenditure user_incentive_increase tokenomics voting_mechanism_change


* Label variables
label var weighted "\${\it Weighted}_{i,t}\$"
label var quadratic "\${\it Quadratic Voting}_{i,t}\$"
label var ranked_choice "\${\it Ranked Choice}_{i,t}\$"
label var quorum "\${\it Quorum}_{i,t}\$"
label var multi_choices "\${\it Multiple Choices}_{i,t}\$"
label var have_discussion "\${\it Discussion}_{i,t}\$"
label var delegation "\${\it Delegation}_{i,t}\$"
label var professionalism  "\${\it Professionalism}_{i,t}\$"
label var concensus       "\${\it Concensus}_{i,t}\$"
label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"
label var whale_user      "\${\it User}_{i,t}^{\it Whale}\$"
label var voter_user      "\${\it User}_{i,t}\$"


. 
. import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(58 vars, 2,830 obs)

. 
. * Parse date
. gen day = date(date, "YMD")

. format day %td

. gen month = month(day)

. gen quarter = quarter(day)

. 
. gen year = year(day)

. encode gecko_id, gen(token)

. encode space,    gen(dao)

. 
. * Dependent variables
. local dep non_whale_turnout whale_turnout

. 
. replace concensus = concensus * have_discussion
(219 real changes made)

. replace professionalism = professionalism * have_discussion
(218 real changes made)

. 
. * Independent variables
. local complexity multi_choices weighted ranked_choice quorum delegation

. local discussion_char concensus professionalism

. local topic protocol_security treasury_expenditure user_incentive_increase to
> kenomics voting_mechanism_change

. 
. 
. * Label variables
. label var weighted "\${\it Weighted}_{i,t}\$"

. label var quadratic "\${\it Quadratic Voting}_{i,t}\$"

. label

## Proposal Characteristics

In [ ]:
%%stata

******************************************************
* PANEL REGRESSIONS
******************************************************

eststo clear

foreach d of local dep {

    * VIF test
    qui reg `d' `discussion' `dapp' `complexity' `topic'
    estat vif

    * Run baseline regression
    reghdfe `d' `complexity' have_discussion `dapp', absorb(year token)
    eststo `d'
    estadd local fe_token "Y"
    estadd local fe_time  "Y"

    * Run regression with discussion characteristics    
    reghdfe `d' `complexity' `discussion_char', absorb(year token)
    eststo `d'_d
    estadd local fe_token "Y"
    estadd local fe_time  "Y"

    * Run regression with user characteristics
    * if it is non_whale_participation, we add non_whale_user; if it is whale_participation, we add whale_user
    if "`d'" == "non_whale_turnout" {
        reghdfe `d' `complexity' have_discussion voter_user, absorb(year token)
        eststo `d'_u
        estadd local fe_token "Y"
        estadd local fe_time  "Y"
    }
    else if "`d'" == "whale_turnout" {
        reghdfe `d' `complexity' have_discussion voter_user, absorb(year token)
        eststo `d'_u
        estadd local fe_token "Y"
        estadd local fe_time  "Y"
    }
}

* Export LaTeX table
esttab                                                           ///
    non_whale_turnout non_whale_turnout_d non_whale_turnout_u    ///
    whale_turnout whale_turnout_d whale_turnout_u                ///
    using "$TABLES/reg_turnout_char_.tex", replace          ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs                                             ///
    b(%9.3f) se(%9.2f)                                           ///
    mgroups("\${\it Turnout}^{Small}_{i,t}\$"              ///
            "\${\it Turnout}^{Whale}_{i,t}\$",             ///
            span                                                 ///
            pattern(1 0 0 1 0 0)                                 ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    mtitles("Full" "Discussion $\geq$ 2" "Have Dapp"             ///
            "Full" "Discussion $\geq$ 2" "Have Dapp")          ///
    posthead("\cmidrule(lr){2-4}\cmidrule(lr){5-7} &\multicolumn{1}{c}{(1)}&\multicolumn{1}{c}{(2)}&\multicolumn{1}{c}{(3)}&\multicolumn{1}{c}{(4)}&\multicolumn{1}{c}{(5)}&\multicolumn{1}{c}{(6)}\\\midrule") ///
    substitute("\_" "_")                                         ///
    stats(fe_token fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Token FE" "Year FE" "Observations" "Adjusted R²"))


. 
. ******************************************************
. * PANEL REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
. foreach d of local dep {
  2. 
.     * VIF test
.     qui reg `d' `discussion' `dapp' `complexity' `topic'
  3.     estat vif
  4. 
.     * Run baseline regression
.     reghdfe `d' `complexity' have_discussion `dapp', absorb(year token)
  5.     eststo `d'
  6.     estadd local fe_token "Y"
  7.     estadd local fe_time  "Y"
  8. 
.     * Run regression with discussion characteristics    
.     reghdfe `d' `complexity' `discussion_char', absorb(year token)
  9.     eststo `d'_d
 10.     estadd local fe_token "Y"
 11.     estadd local fe_time  "Y"
 12. 
.     * Run regression with user characteristics
.     * if it is non_whale_participation, we add non_whale_user; if it is whale
> _participation, we add whale_user
.     if "`d'" == "non_whale_turnout" {
 13.         reghdfe `d' `complexity' have_discussion voter_user, absor